# 📱 FeatureIQ: Consumer Smartphone Preferences Analytics
**Business Analytics Individual Case Study**

---

**Author:** Nishanth  
**Dataset:** `final_dataset.csv` (10,000 synthesized market preference profiles)  

### 🎯 Executive Summary
Smartphone manufacturers often struggle to align hardware components (RAM, Camera) with what consumers actually value within specific price brackets. In this study, we:
1. **Explore Market Segmentation** to understand feature prioritization across Budget, Mid-Range, and Premium demographics.
2. **Train a Random Forest Classifier** to map complex non-linear specifications directly to market segments.
3. **Extract Feature Importances** to give R&D departments actionable guidance on where to spend manufacturing budgets.

Let's explore the data interactively!

## 🛠️ 1. Data Preparation & Exploration
First, we import our standard data manipulation libraries and `plotly` for rich, interactive visualizations.

In [7]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

import warnings
warnings.filterwarnings('ignore')

# Load the Dataset
df = pd.read_csv('data/final_dataset.csv')
print(f"Market Data Shape: {df.shape}")
display(df.head())

Market Data Shape: (10000, 13)


,Smartphone_Model,Brand,Battery_mAh,RAM_GB,Storage_GB,Camera_MP,Screen_Size_inches,Budget_Segment,Pref_Battery,Pref_Camera,Pref_Performance,Pref_Brand,Price_USD
0,Phone_0,OnePlus,4000,6,128,48,6.5,Budget,7,5,4,5,127.388056
1,Phone_1,Oppo,5000,4,128,50,6.5,Budget,9,3,4,2,224.855001
2,Phone_2,Xiaomi,6000,4,512,12,6.5,Mid-Range,5,6,7,6,599.645415
3,Phone_3,Oppo,5000,4,128,48,6.7,Budget,8,4,3,2,156.479634
4,Phone_4,Oppo,5000,8,512,12,6.8,Mid-Range,8,6,8,4,694.627396


### 📊 Interactive Exploratory Data Analysis (EDA)
How are our consumers distributed across different price and specification segments?

In [8]:
fig = px.histogram(df, x='Budget_Segment', color='Budget_Segment', 
                   title='Market Share by Budget Segment',
                   color_discrete_sequence=px.colors.qualitative.Bold,
                   text_auto=True)
fig.update_layout(showlegend=False, xaxis_title="Segment", yaxis_title="Consumer Volume")
fig.show()

In [9]:
# Melting the preference columns to create a grouped bar chart
pref_cols = ['Pref_Battery', 'Pref_Camera', 'Pref_Performance', 'Pref_Brand']
avg_prefs = df.groupby('Budget_Segment')[pref_cols].mean().reset_index()
avg_prefs_melted = avg_prefs.melt(id_vars='Budget_Segment', var_name='Feature', value_name='Average_Score')

fig = px.bar(avg_prefs_melted, x='Feature', y='Average_Score', color='Budget_Segment', barmode='group',
             title='Average Feature Preference Scores by Segment',
             color_discrete_sequence=px.colors.qualitative.Prism)
fig.show()

In [13]:
fig = px.scatter(df.sample(1000, random_state=42), x='RAM_GB', y='Storage_GB', color='Budget_Segment', 
                 size='Price_USD', hover_data=['Brand'], 
                 title='Hardware Specs Correlation to Budget Segment (Sampled)',
                 opacity=0.7)
fig.show()

## 🤖 2. Machine Learning: Random Forest Classification
Hardware specs and consumer preferences aren't linear—they have strict tiering rules (e.g., a "Premium" phone *must* have >8GB RAM). Therefore, a **Random Forest Classifier** is the perfect algorithm to map these non-linear thresholds to budget segments.

In [11]:
X = df[['Pref_Battery', 'Pref_Camera', 'Pref_Performance', 'Pref_Brand', 
        'Battery_mAh', 'RAM_GB', 'Storage_GB', 'Camera_MP', 'Screen_Size_inches']]
y = df['Budget_Segment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(n_estimators=150, random_state=42, max_depth=12)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"⭐ Random Forest Model Accuracy: {acc*100:.2f}%\n")
print(classification_report(y_test, y_pred))

⭐ Random Forest Model Accuracy: 100.00%

              precision    recall  f1-score   support

      Budget       1.00      1.00      1.00       291
   Mid-Range       1.00      1.00      1.00      1200
     Premium       1.00      1.00      1.00       509

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



### 🧠 Interpreting the Model (Feature Importance)
By looking inside the "black box" of the Random Forest, we can tell R&D exactly which components move a device from "Mid-Range" to "Premium" in the eyes of the consumer.

In [12]:
feature_importances = pd.DataFrame({'Feature': X.columns, 'Importance': rf_model.feature_importances_})
feature_importances = feature_importances.sort_values('Importance', ascending=True)

fig = px.bar(feature_importances, x='Importance', y='Feature', orientation='h',
             title='Feature Importance for Pricing Power & Segmentation',
             color='Importance', color_continuous_scale='Magma')
fig.show()

## 💼 3. Business Recommendations & Conclusion

Based on our EDA and Machine Learning model, the following business actions are recommended:

1. **R&D Allocation for Budget Devices:** The "Budget" segment strongly indexes on `Pref_Battery`. Manufacturers should prioritize massive batteries (e.g., 6000mAh) over high megapixel cameras, as high-end cameras do not drive sales in this segment.
2. **The Premium Threshold:** The Random Forest proved that `Storage_GB` and `RAM_GB` are the highest deterministic features for moving a device into the "Premium" category. A device cannot successfully command a flagship price without these specs maximized.
3. **Segmented Advertising:** Stop marketing "all day battery life" to the Premium segment. They care about raw performance and camera optics. Marketing copy must align with the specific feature importances of the targeted demographic.